<a href="https://colab.research.google.com/github/faaiz-ai/New-Practice-/blob/main/CNN_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


In [2]:
import os

os.environ["KAGGLE_USERNAME"] = "faaiznawaz"
os.environ["KAGGLE_API_TOKEN"] = "KGAT_773b76fbb5536242e0c1870d54b73a05"

In [3]:
!unzip archive(8).zip

/bin/bash: -c: line 1: syntax error near unexpected token `('
/bin/bash: -c: line 1: `unzip archive(8).zip'


In [4]:
!kaggle datasets download -d bhavikjikadara/dog-and-cat-classification-dataset

Dataset URL: https://www.kaggle.com/datasets/bhavikjikadara/dog-and-cat-classification-dataset
License(s): apache-2.0
100% 775M/775M [00:07<00:00, 110MB/s]



In [15]:
!unzip -o dog-and-cat-classification-dataset.zip -d . > /dev/null

In [16]:
import os

print(os.listdir("PetImages"))

print("Number of Cat images:", len(os.listdir("PetImages/Cat")))
print("Number of Dog images:", len(os.listdir("PetImages/Dog")))

['Dog', 'Cat']
Number of Cat images: 12499
Number of Dog images: 12499


In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [8]:
import shutil
from sklearn.model_selection import train_test_split

original_data_dir = "PetImages"
base_dir = "data"

# Create folders
for split in ["train", "val", "test"]:
    for cls in ["Cat", "Dog"]:
        os.makedirs(os.path.join(base_dir, split, cls), exist_ok=True)

# Split ratios
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

def split_class(cls):
    cls_dir = os.path.join(original_data_dir, cls)
    images = [f for f in os.listdir(cls_dir) if f.endswith((".jpg", ".jpeg", ".png"))]

    # Step 1: test split
    train_val, test = train_test_split(images, test_size=test_ratio, random_state=42)

    # Step 2: validation split
    train, val = train_test_split(train_val, test_size=val_ratio/(train_ratio+val_ratio), random_state=42)

    # Step 3: copy files
    for img in train:
        shutil.copy(os.path.join(cls_dir, img), os.path.join(base_dir, "train", cls, img))
    for img in val:
        shutil.copy(os.path.join(cls_dir, img), os.path.join(base_dir, "val", cls, img))
    for img in test:
        shutil.copy(os.path.join(cls_dir, img), os.path.join(base_dir, "test", cls, img))

for cls in ["Cat","Dog"]:
    split_class(cls)

print("Dataset cleaned and split successfully!")

Dataset cleaned and split successfully!


In [9]:
transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5], [0.5,0.5,0.5])
])

train_dataset = datasets.ImageFolder(os.path.join(base_dir, "train"), transform=transform)
val_dataset   = datasets.ImageFolder(os.path.join(base_dir, "val"), transform=transform)
test_dataset  = datasets.ImageFolder(os.path.join(base_dir, "test"), transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [10]:
import os

base_dir = "data"  # make sure this is the folder you used
for split in ["train", "val", "test"]:
    for cls in ["Cat", "Dog"]:
        folder_path = os.path.join(base_dir, split, cls)
        print(f"{folder_path}: {len(os.listdir(folder_path))} files")

data/train/Cat: 8749 files
data/train/Dog: 8749 files
data/val/Cat: 1875 files
data/val/Dog: 1875 files
data/test/Cat: 1875 files
data/test/Dog: 1875 files


In [11]:
dataiter = iter(train_loader)
images, labels = next(dataiter)
print(images.shape)
print(labels[:10])

torch.Size([32, 3, 128, 128])
tensor([1, 0, 0, 0, 1, 0, 0, 1, 1, 1])


In [12]:
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.fc1 = nn.Linear(32 * 16 * 16, 128)
        self.fc2 = nn.Linear(128, 2)
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 32*16*16)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = SimpleCNN()

In [13]:
criterion = nn.CrossEntropyLoss()  # for multi-class classification
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
epochs = 5

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader):.4f}")

ValueError: Expected input batch_size (128) to match target batch_size (32).

In [ ]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Validation Accuracy: {100 * correct / total:.2f}%")